### Model Loading and Prediction
- Here, we will load the model that we have saved in the previous step and use it to make predictions on test data. This is how it will be done in a real world scenario, where we will have a trained model and we will use it to make predictions on new data. We will also check the accuracy of the model on test data.

In [12]:
import pandas as pd
import numpy as np
import zipfile
from matplotlib import pyplot as py
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold,GridSearchCV,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import time
from sklearn.metrics import accuracy_score,roc_auc_score,confusion_matrix,classification_report,f1_score,precision_score,recall_score
import joblib

In [2]:
loadedModel = joblib.load("IBMTelecoCustomerChurnModel.pkl")

In [9]:
#Personal lap path
zip_path = r"D:\Veena\Certificate Program in AI and ML\Dataset.zip"
#office lap path
#zip_path = r"E:\Veena\Certificate-Program-in-AIML-IIT-Patna\MODULE 2 - CLASSICAL ML\Dataset.zip"
with zipfile.ZipFile(zip_path) as z:
    ibmTelecoChurnDF = pd.read_csv(z.open("Dataset/telco_customer_churn.csv"))


### Fixing the data before prediction

In [10]:
#Converting the obtained boolean value(Yes-> 1, No-> 0) from ibmTelecoChurnDF['Churn'] == 'Yes', to integer type as 1 and 0.
ibmTelecoChurnDF['ChurnInt'] = (ibmTelecoChurnDF['Churn'] == 'Yes').astype(int)
#Convert "Total Charges" column into numeric
ibmTelecoChurnDF['TotalCharges'] = pd.to_numeric(ibmTelecoChurnDF['TotalCharges'],errors='coerce')
ibmTelecoChurnDF['TotalCharges']=ibmTelecoChurnDF['TotalCharges'].fillna(0)

#Droping the unnecessary column - customer ID
ibmTelecoChurnDF = ibmTelecoChurnDF.drop('customerID',axis=1)



### Test Train Split
- Just because I am going to predict using xTest, I did the below step. If I am directly going to predict on new customer data, I will not do this step. I will directly use the new customer data for prediction. But here, I am using xTest to predict and check the accuracy of the model on test data. So, I need to split the data into train and test sets.

In [13]:
x = ibmTelecoChurnDF.drop(['Churn','ChurnInt'],axis=1)
y = ibmTelecoChurnDF['ChurnInt']

xTrain,xTest,yTrain,yTest = train_test_split(x,y,test_size=0.25,random_state=42,stratify=y)

print(f"Train samples : {len(xTrain)} samples | Churn Rate: {yTrain.mean():.2%}")
print(f"Test samples : {len(xTest)} samples | Churn Rate: {yTest.mean():.2%}")

Train samples : 5282 samples | Churn Rate: 26.54%
Test samples : 1761 samples | Churn Rate: 26.52%


### Prediction

In [14]:
yPredTest = loadedModel.predict(xTest)
yPredTest

array([0, 0, 0, ..., 0, 1, 0], shape=(1761,))

In [15]:
xTest.iloc[5]

gender                        Female
SeniorCitizen                      1
Partner                           No
Dependents                        No
tenure                            16
PhoneService                     Yes
MultipleLines                    Yes
InternetService          Fiber optic
OnlineSecurity                    No
OnlineBackup                      No
DeviceProtection                  No
TechSupport                       No
StreamingTV                      Yes
StreamingMovies                  Yes
Contract              Month-to-month
PaperlessBilling                 Yes
PaymentMethod       Electronic check
MonthlyCharges                  96.4
TotalCharges                  1581.2
Name: 2516, dtype: object

In [16]:
yPredTest[5]

np.int64(1)

In [23]:
yTest.iloc[5]

np.int64(1)

In [33]:
newCustomers = pd.DataFrame([
    {
        'gender': 'Male',
        'SeniorCitizen': 0,
        'Partner': 'Yes',
        'Dependents': 'No',
        'tenure': 48,
        'PhoneService': 'Yes',
        'MultipleLines': 'No',
        'InternetService': 'DSL',
        'OnlineSecurity': 'Yes',
        'OnlineBackup': 'Yes',
        'DeviceProtection': 'Yes',
        'TechSupport': 'Yes',
        'StreamingTV': 'No',
        'StreamingMovies': 'No',
        'Contract': 'Two year',
        'PaperlessBilling': 'No',
        'PaymentMethod': 'Bank transfer (automatic)',
        'MonthlyCharges': 65.50,
        'TotalCharges': 3144.00
    },
    {
        'gender': 'Female',
        'SeniorCitizen': 1,
        'Partner': 'No',
        'Dependents': 'No',
        'tenure': 5,
        'PhoneService': 'Yes',
        'MultipleLines': 'Yes',
        'InternetService': 'Fiber optic',
        'OnlineSecurity': 'No',
        'OnlineBackup': 'No',
        'DeviceProtection': 'No',
        'TechSupport': 'No',
        'StreamingTV': 'Yes',
        'StreamingMovies': 'Yes',
        'Contract': 'Month-to-month',
        'PaperlessBilling': 'Yes',
        'PaymentMethod': 'Electronic check',
        'MonthlyCharges': 98.40,
        'TotalCharges': 492.00
    },
    {
        'gender': 'Male',
        'SeniorCitizen': 0,
        'Partner': 'No',
        'Dependents': 'No',
        'tenure': 18,
        'PhoneService': 'Yes',
        'MultipleLines': 'No',
        'InternetService': 'Fiber optic',
        'OnlineSecurity': 'No',
        'OnlineBackup': 'Yes',
        'DeviceProtection': 'No',
        'TechSupport': 'No',
        'StreamingTV': 'Yes',
        'StreamingMovies': 'No',
        'Contract': 'One year',
        'PaperlessBilling': 'Yes',
        'PaymentMethod': 'Credit card (automatic)',
        'MonthlyCharges': 84.75,
        'TotalCharges': 1525.50
    },
    {
        'gender': 'Female',
        'SeniorCitizen': 0,
        'Partner': 'Yes',
        'Dependents': 'Yes',
        'tenure': 72,
        'PhoneService': 'Yes',
        'MultipleLines': 'Yes',
        'InternetService': 'DSL',
        'OnlineSecurity': 'Yes',
        'OnlineBackup': 'Yes',
        'DeviceProtection': 'Yes',
        'TechSupport': 'Yes',
        'StreamingTV': 'Yes',
        'StreamingMovies': 'Yes',
        'Contract': 'Two year',
        'PaperlessBilling': 'No',
        'PaymentMethod': 'Mailed check',
        'MonthlyCharges': 89.20,
        'TotalCharges': 6422.40
    },
    {
        'gender': 'Male',
        'SeniorCitizen': 1,
        'Partner': 'No',
        'Dependents': 'No',
        'tenure': 2,
        'PhoneService': 'Yes',
        'MultipleLines': 'No',
        'InternetService': 'Fiber optic',
        'OnlineSecurity': 'No',
        'OnlineBackup': 'No',
        'DeviceProtection': 'No',
        'TechSupport': 'No',
        'StreamingTV': 'No',
        'StreamingMovies': 'No',
        'Contract': 'Month-to-month',
        'PaperlessBilling': 'Yes',
        'PaymentMethod': 'Electronic check',
        'MonthlyCharges': 79.90,
        'TotalCharges': 159.80
    }
])

In [34]:
newCustomers

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,0,Yes,No,48,Yes,No,DSL,Yes,Yes,Yes,Yes,No,No,Two year,No,Bank transfer (automatic),65.50,3144.0
1,Female,1,No,No,5,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.40,492.0
2,Male,0,No,No,18,Yes,No,Fiber optic,No,Yes,No,No,Yes,No,One year,Yes,Credit card (automatic),84.75,1525.5
3,Female,0,Yes,Yes,72,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,No,Mailed check,89.20,6422.4
4,Male,1,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,79.90,159.8


In [35]:
newCustomerPredicTion = loadedModel.predict(newCustomers)

In [36]:
newCustomerPredicTion

array([0, 1, 0, 0, 1])